In [1]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np
import glob
import pickle
from collections import defaultdict
from pprint import pprint

In [1]:
file_dir = 'spaceinvaderscheckpoint/spaceinvaders_10e6_2.npy'
file_dir = 'thinker-20250806-002733-320-0/video_stat.npy'
file_dir = 'thinker-20251023-165530-327280-0/video_stat.npy'
file_dir = 'test/spaceinvaders_4e6_1.npy'
data_dir = os.path.join('./test', file_dir)
data = (np.load(data_dir, allow_pickle=True)).item()
print('data.keys():', data.keys())
print('real image data length:', len(data['real_imgs']))
print('imagination image data length:', len(data['im_imgs']))

NameError: name 'os' is not defined

In [ ]:
# data['status']에서 각 값의 개수 계산
status_values = [0, 1, 2, 3]  # 계산하려는 status 값들
status_counts = {}

for value in status_values:
    # 해당 값을 가진 인덱스의 개수 계산
    count = np.sum(np.array(data['status']) == value)
    status_counts[value] = count

# 결과 출력
for value, count in status_counts.items():
    status_name = {
        0: "실제 스텝(real step)",
        1: "리셋(reset)",
        2: "상상 스텝(imaginary step)",
        3: "강제 리셋(force reset)"
    }.get(value, "알 수 없음")
    
    print(f"Status {value} ({status_name}): {count}개")

# 특정 값(예: 0)을 가진 인덱스 목록 구하기
indices_with_value_1 = np.where(np.array(data['status']) == 1)[0]
print(f"\nStatus 1을 가진 인덱스의 개수: {len(indices_with_value_1)}")

In [ ]:
def analyze_noop_planning_relationship(data, window_size=10, stride=2):
    """
    Sliding window를 사용하여 NOOP action 빈도와 planning depth의 관계를 분석합니다.
    status가 0인 real step 데이터만 분석 대상으로 합니다.
    
    Args:
        data: 로드된 데이터 딕셔너리
        window_size: sliding window 크기 (real step 기준)
        stride: sliding 간격 (real step 기준)
    
    Returns:
        window_results: 각 윈도우별 분석 결과
    """
    status_data = data['status']
    total_length = len(status_data)
    
    # status 데이터 분석 (planning depth 계산용) - 새로운 방식 사용
    real_indices, planning_depths, reset_info = analyze_status_data(status_data)
    
    # NOOP action 추출 (one-hot encoding 처리)
    noop_actions = []
    for idx in range(total_length):
        if idx < len(data['tree_reps']['cur_action']):
            action_onehot = data['tree_reps']['cur_action'][idx]
            action_idx = np.argmax(action_onehot)
            noop_actions.append(action_idx)
        else:
            noop_actions.append(-1)  # 데이터가 없는 경우
    
    noop_actions = np.array(noop_actions)
    
    # status가 0인 real step에서만 NOOP action 추출
    real_step_noop_actions = []
    for idx in real_indices:
        if idx < len(noop_actions):
            real_step_noop_actions.append(noop_actions[idx])
        else:
            real_step_noop_actions.append(-1)
    
    real_step_noop_actions = np.array(real_step_noop_actions)
    
    window_results = []
    
    # Real step 기준으로 sliding window 분석
    for start_idx in range(0, len(real_indices) - window_size + 1, stride):
        end_idx = start_idx + window_size
        
        # 현재 윈도우의 real step 인덱스들
        window_real_indices = real_indices[start_idx:end_idx]
        window_noop_actions = real_step_noop_actions[start_idx:end_idx]
        window_planning_depths = planning_depths[start_idx:end_idx]
        
        # 1. NOOP action 분석 (real step에서만)
        noop_count = np.sum(window_noop_actions == 0)  # NOOP = 0
        noop_frequency = noop_count / window_size
        
        # 2. Planning depth 분석 - 윈도우 내 모든 real step의 planning depth 평균
        avg_planning_depth = np.mean(window_planning_depths)
        
        # 3. 결과 저장
        window_result = {
            'window_start': start_idx,
            'window_end': end_idx,
            'real_step_indices': window_real_indices,
            'noop_count': noop_count,
            'noop_frequency': noop_frequency,
            'avg_planning_depth': avg_planning_depth,
            'planning_depths': window_planning_depths,
            'noop_actions': window_noop_actions
        }
        
        window_results.append(window_result)
    
    return window_results

# 데이터 로드 및 분석
status_data = data['status']
real_indices, planning_depths, reset_info = analyze_status_data(status_data)

print(f"총 데이터 길이: {len(status_data)}")
print(f"Real step 개수: {len(real_indices)}")

if planning_depths:
    print(f"Planning Depth 통계 (각 real step별 평균값):")
    print(f"  - 평균: {np.mean(planning_depths):.2f}")
    print(f"  - 중앙값: {np.median(planning_depths):.2f}")
    print(f"  - 최대값: {np.max(planning_depths):.2f}")
    print(f"  - 최소값: {np.min(planning_depths):.2f}")
    print(f"  - 표준편차: {np.std(planning_depths):.2f}")
else:
    print("Planning depth 데이터가 없습니다.")

# 각 real step마다의 구간 수 통계
segment_counts = [len(info['segment_lengths']) for info in reset_info]
print(f"\n각 real step마다의 구간 수 통계:")
print(f"  - 평균: {np.mean(segment_counts):.2f}")
print(f"  - 중앙값: {np.median(segment_counts):.2f}")
print(f"  - 최대값: {np.max(segment_counts)}")
print(f"  - 최소값: {np.min(segment_counts)}")
print(f"  - 표준편차: {np.std(segment_counts):.2f}")

# 예시 출력 (처음 몇 개 real step의 planning depth)
print(f"\n=== 예시: 처음 5개 real step의 planning depth ===")
for i in range(min(5, len(planning_depths))):
    print(f"Real step {i+1} (인덱스 {real_indices[i]}): "
          f"Planning Depth = {planning_depths[i]:.2f}, "
          f"구간들 = {reset_info[i]['segment_lengths']}")

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns

action_mapping = {
    'NOOP': 0,
    'FIRE': 1,
    'RIGHT': 2,
    'LEFT': 3,
    'RIGHTFIRE': 4,
    'LEFTFIRE': 5,
}
idx_to_action = {v: k for k, v in action_mapping.items()}

def vectorize_tree_reps(tree_reps, fields=None):
    if fields is None:
        fields = [k for k, v in tree_reps.items() if isinstance(v, np.ndarray)]
    lengths = [tree_reps[k].shape[0] for k in fields]
    common_len = min(lengths)
    vectors = []
    for key in fields:
        arr = np.asarray(tree_reps[key][:common_len])
        vectors.append(arr.reshape(common_len, -1))
    return np.concatenate(vectors, axis=1), common_len

def prepare_real_step_tree_matrix(data):
    status_data = data['status']
    real_indices, planning_depths, reset_info = analyze_status_data(status_data)

    tree_matrix, max_len = vectorize_tree_reps(data['tree_reps'])
    real_indices = np.array(real_indices)
    valid = real_indices < tree_matrix.shape[0]
    real_indices = real_indices[valid]
    tree_real = tree_matrix[real_indices]

    action_ids = np.argmax(data['tree_reps']['cur_action'][:max_len], axis=1)
    action_labels = np.array([idx_to_action.get(a, f"A{a}") for a in action_ids])
    action_labels = action_labels[real_indices]

    planning_depths = np.array(planning_depths)[valid]

    return tree_real, action_labels, planning_depths, real_indices

def plot_tree_rep_tsne(data, perplexity=30, fields=None, random_state=0):
    tree_real, action_labels, planning_depths, real_indices = prepare_real_step_tree_matrix(data)

    scaler = StandardScaler()
    tree_scaled = scaler.fit_transform(tree_real)

    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        random_state=random_state,
        init='pca',
        learning_rate='auto'
    )
    emb = tsne.fit_transform(tree_scaled)

    tsne_df = {
        'tsne_1': emb[:, 0],
        'tsne_2': emb[:, 1],
        'action': action_labels,
        'planning_depth': planning_depths,
        'real_idx': real_indices,
    }

    depth_style = np.where(planning_depths > np.median(planning_depths), 'High depth', 'Low depth')

    plt.figure(figsize=(9, 6))
    sns.scatterplot(
        x=tsne_df['tsne_1'],
        y=tsne_df['tsne_2'],
        hue=tsne_df['action'],
        palette='tab10',
        alpha=0.75,
        edgecolor='none'
    )
    plt.title('Tree Representation t-SNE (action-wise)')
    plt.xlabel('t-SNE 1')
    plt.ylabel('t-SNE 2')
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.)
    plt.tight_layout()
    plt.show()

    return tsne_df

# 예시 실행
tsne_results = plot_tree_rep_tsne(data, perplexity=35)

In [4]:
"""Utilities for visualizing behavioral embeddings across checkpoints.

These helpers iterate over every ``.npy`` file inside a folder, extract the
real-step representations for a chosen source (``tree_reps`` dict entries,
top-level ``real_imgs`` arrays, latent ``real_vectors``, …), and render t‑SNE
plots where each action id owns a hue while the training step is encoded as
brightness.  You can also batch multiple targets (tree_reps vs. raw frames vs.
latent vectors) in a single call.
"""

from __future__ import annotations

import math
import os
import re
from pathlib import Path
from typing import List, Optional, Sequence, Tuple, Union

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# ---------------------------------------------------------------------------
# Shared constants/helpers
# ---------------------------------------------------------------------------

ACTION_MAPPING = {
    "NOOP": 0,
    "FIRE": 1,
    "RIGHT": 2,
    "LEFT": 3,
    "RIGHTFIRE": 4,
    "LEFTFIRE": 5,
}
IDX_TO_ACTION = {v: k for k, v in ACTION_MAPPING.items()}

TRAINING_STEP_IN_NAME = re.compile(r"_(\d+(?:e\d+)?)(?=(?:_\d+)?\.npy$)", re.IGNORECASE)
TRAINING_STEP_IN_DIR = re.compile(r"(\d+(?:e\d+)?)$", re.IGNORECASE)
ALL_FIELD_SENTINELS = {"tree_reps", "all", "ALL", "*"}


def _format_training_step(step: float) -> str:
    """Render training step as a short human-readable string."""
    if step >= 1_000_000:
        return f"{step/1_000_000:.1f}M"
    if step >= 1_000:
        return f"{step/1_000:.1f}K"
    return f"{int(step)}"


# ---------------------------------------------------------------------------
# Status / tree_rep processing
# ---------------------------------------------------------------------------

def analyze_status_data(status_data: Sequence[int]) -> Tuple[List[int], List[float], List[dict]]:
    """Replicates the real-step/planning-depth extraction used in the notebooks."""
    real_indices: List[int] = []
    planning_depths: List[float] = []
    reset_info: List[dict] = []

    i = 0
    while i < len(status_data):
        if status_data[i] == 0:
            real_indices.append(i)
            imagination_start = i + 1
            imagination_end = len(status_data)
            for j in range(i + 1, len(status_data)):
                if status_data[j] == 0:
                    imagination_end = j
                    break

            if imagination_start < imagination_end:
                imagination_segment = status_data[imagination_start:imagination_end]
                segment_lengths: List[int] = []
                current_length = 0
                reset_positions: List[int] = []

                for k, status_val in enumerate(imagination_segment):
                    if status_val in (1, 3):
                        if current_length > 0:
                            segment_lengths.append(current_length)
                        reset_positions.append(imagination_start + k)
                        current_length = 0
                    elif status_val == 2:
                        current_length += 1

                if current_length > 0:
                    segment_lengths.append(current_length)

                avg_planning_depth = float(np.mean(segment_lengths)) if segment_lengths else 0.0
                planning_depths.append(avg_planning_depth)
                reset_info.append(
                    {
                        "reset_positions": reset_positions,
                        "segment_lengths": segment_lengths,
                        "total_length": sum(segment_lengths),
                        "avg_planning_depth": avg_planning_depth,
                    }
                )
            else:
                planning_depths.append(0.0)
                reset_info.append(
                    {
                        "reset_positions": [],
                        "segment_lengths": [],
                        "total_length": 0,
                        "avg_planning_depth": 0.0,
                    }
                )

            i = imagination_end
        else:
            i += 1

    return real_indices, planning_depths, reset_info


def _normalize_tree_fields(
    tree_reps: dict, tree_fields: Optional[Union[str, Sequence[str]]]
) -> List[str]:
    """Resolve which tree_rep arrays to use."""
    if tree_fields is None:
        return [k for k, v in tree_reps.items() if isinstance(v, np.ndarray)]

    if isinstance(tree_fields, str):
        if tree_fields in ALL_FIELD_SENTINELS:
            return _normalize_tree_fields(tree_reps, None)
        tree_fields = [tree_fields]

    filtered_fields = [f for f in tree_fields if f not in ALL_FIELD_SENTINELS]
    if not filtered_fields:
        return _normalize_tree_fields(tree_reps, None)

    resolved = []
    missing = []
    for field in filtered_fields:
        if field in tree_reps:
            resolved.append(field)
        else:
            missing.append(field)

    if missing:
        raise KeyError(f"Missing tree_rep fields: {missing}")

    return resolved


def vectorize_tree_reps(
    tree_reps: dict, tree_fields: Optional[Union[str, Sequence[str]]] = None
) -> Tuple[np.ndarray, int]:
    """Stack/flatten the requested tree_rep arrays into a 2‑D matrix."""
    fields = _normalize_tree_fields(tree_reps, tree_fields)
    lengths = [tree_reps[k].shape[0] for k in fields]
    common_len = int(min(lengths))

    vectors = []
    for key in fields:
        arr = np.asarray(tree_reps[key][:common_len])
        vectors.append(arr.reshape(common_len, -1))

    return np.concatenate(vectors, axis=1), common_len


def _flatten_step_array(arr: np.ndarray, max_len: Optional[int] = None) -> Tuple[np.ndarray, int]:
    """Flatten per-step tensors into a (steps, feature_dim) matrix."""
    arr = np.asarray(arr)
    if max_len is not None:
        arr = arr[:max_len]
    if arr.ndim == 1:
        flat = arr.reshape(arr.shape[0], 1)
    else:
        flat = arr.reshape(arr.shape[0], -1)
    return flat, flat.shape[0]


def prepare_real_step_matrix(
    data: dict,
    source: str = "tree_reps",
    tree_fields: Optional[Union[str, Sequence[str]]] = None,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Return vectors, action ids, planning depths, and real indices for real steps."""
    status_data = np.asarray(data["status"])
    real_indices, planning_depths, _ = analyze_status_data(status_data)

    if len(real_indices) == 0:
        raise ValueError("No real steps (status == 0) found in status data.")

    cur_action_full = np.asarray(data["tree_reps"]["cur_action"])

    if source == "tree_reps":
        matrix, common_len = vectorize_tree_reps(data["tree_reps"], tree_fields=tree_fields)
    else:
        if source not in data:
            raise KeyError(f"Source '{source}' not present in data keys: {list(data.keys())}")
        limit = min(len(data[source]), len(cur_action_full))
        matrix, common_len = _flatten_step_array(data[source], max_len=limit)

    cur_action = cur_action_full[:common_len]
    action_ids = np.argmax(cur_action, axis=1)

    real_indices = np.asarray(real_indices)
    valid = real_indices < matrix.shape[0]
    if not np.any(valid):
        raise ValueError(f"Real indices are outside the available range for source '{source}'.")

    real_indices = real_indices[valid]
    features = matrix[real_indices]
    action_ids = action_ids[real_indices]
    planning_depths = np.asarray(planning_depths)[valid]

    return features, action_ids, planning_depths, real_indices


# ---------------------------------------------------------------------------
# Dataset aggregation
# ---------------------------------------------------------------------------

def extract_training_step(path_like: Union[str, os.PathLike]) -> int:
    """Infer the training/global step from a checkpoint filename or folder."""
    path = Path(path_like)
    name_match = TRAINING_STEP_IN_NAME.search(path.name)
    if name_match:
        token = name_match.group(1)
        try:
            return int(float(token))
        except ValueError:
            pass

    dir_match = TRAINING_STEP_IN_DIR.search(path.parent.name)
    if dir_match:
        token = dir_match.group(1)
        try:
            return int(float(token))
        except ValueError:
            pass

    for part in reversed(path.parent.parts):
        if part.startswith("thinker-"):
            chunks = part.split("-")
            for chunk in reversed(chunks):
                if chunk.isdigit() or TRAINING_STEP_IN_DIR.match(chunk or ""):
                    try:
                        return int(float(chunk))
                    except ValueError:
                        continue
    return 0


def build_tree_rep_dataset(
    root_folder: Union[str, os.PathLike],
    tree_fields: Optional[Union[str, Sequence[str]]] = "tree_reps",
    source: str = "tree_reps",
    max_files: Optional[int] = None,
    stride: int = 1,
    logger=print,
) -> dict:
    """Load every ``.npy`` file in a folder and aggregate real-step vectors."""
    folder = Path(root_folder)
    npy_files = sorted(folder.glob("*.npy"))
    if not npy_files:
        raise FileNotFoundError(f"No .npy files found under {folder}")

    all_vectors: List[np.ndarray] = []
    all_actions: List[np.ndarray] = []
    all_labels: List[np.ndarray] = []
    all_planning: List[np.ndarray] = []
    all_steps: List[np.ndarray] = []
    all_sources: List[np.ndarray] = []
    all_real_idx: List[np.ndarray] = []

    for idx, npy_path in enumerate(npy_files):
        if max_files is not None and idx >= max_files:
            break
        try:
            data = np.load(npy_path, allow_pickle=True).item()
        except Exception as exc:  # noqa: BLE001 - relay to caller
            if logger:
                logger(f"[skip] Failed to load {npy_path.name}: {exc}")
            continue

        try:
            tree_real, action_ids, planning_depths, real_indices = prepare_real_step_matrix(
                data, source=source, tree_fields=tree_fields
            )
        except Exception as exc:  # noqa: BLE001
            if logger:
                logger(f"[skip] Failed to parse {npy_path.name}: {exc}")
            continue

        if stride > 1:
            tree_real = tree_real[::stride]
            action_ids = action_ids[::stride]
            planning_depths = planning_depths[::stride]
            real_indices = real_indices[::stride]

        if tree_real.size == 0:
            continue

        training_step = extract_training_step(npy_path)
        step_arr = np.full(tree_real.shape[0], training_step, dtype=np.int64)
        source_arr = np.array([npy_path.name] * tree_real.shape[0])
        action_labels = np.vectorize(lambda a: IDX_TO_ACTION.get(int(a), f"A{int(a)}"))(action_ids)

        all_vectors.append(tree_real)
        all_actions.append(action_ids)
        all_labels.append(action_labels)
        all_planning.append(planning_depths)
        all_steps.append(step_arr)
        all_sources.append(source_arr)
        all_real_idx.append(real_indices.astype(np.int64))

        if logger:
            logger(
                f"[loaded] {npy_path.name:<35s} "
                f"real_steps={tree_real.shape[0]:5d}  step={_format_training_step(training_step)}  source={source}"
            )

    if not all_vectors:
        raise RuntimeError(f"No usable tree_reps found under {folder}.")

    def _stack(chunks: List[np.ndarray]) -> np.ndarray:
        return np.concatenate(chunks, axis=0)

    dataset = {
        "tree_matrix": _stack(all_vectors),
        "action_ids": _stack(all_actions),
        "action_labels": _stack(all_labels),
        "planning_depths": _stack(all_planning),
        "training_steps": _stack(all_steps),
        "source_files": _stack(all_sources),
        "real_indices": _stack(all_real_idx),
    }

    return dataset


# ---------------------------------------------------------------------------
# Visualization helpers
# ---------------------------------------------------------------------------

def _mix_with_white(color: Union[str, Sequence[float]], strength: float) -> Tuple[float, float, float]:
    """Blend the given color with white (strength in [0, 1])."""
    rgb = np.asarray(mcolors.to_rgb(color))
    strength = float(np.clip(strength, 0.0, 1.0))
    return tuple(rgb * (1.0 - strength) + strength)


def _compute_point_colors(action_ids: np.ndarray, training_steps: np.ndarray) -> Tuple[List[Tuple[float, float, float]], dict]:
    """Assign a consistent base hue per action and vary brightness by training step."""
    unique_actions = sorted(np.unique(action_ids))
    base_palette = sns.color_palette("tab10", n_colors=max(10, len(unique_actions)))
    base_color_map = {
        action: base_palette[action % len(base_palette)] for action in unique_actions
    }

    min_step = float(training_steps.min())
    max_step = float(training_steps.max())
    if math.isclose(min_step, max_step):
        norm = np.zeros_like(training_steps, dtype=float)
    else:
        norm = (training_steps - min_step) / (max_step - min_step)
    strengths = 0.15 + 0.7 * norm  # keep a bit of the original hue

    colors = []
    for action_id, strength in zip(action_ids, strengths):
        base = base_color_map[action_id]
        colors.append(_mix_with_white(base, float(strength)))

    return colors, base_color_map


def fit_tsne(tree_matrix: np.ndarray, perplexity: int = 30, random_state: int = 0) -> np.ndarray:
    """Scale + fit a t-SNE embedding."""
    scaler = StandardScaler()
    scaled = scaler.fit_transform(tree_matrix)
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        random_state=random_state,
        init="pca",
        learning_rate="auto",
        n_iter=1000,
        verbose=1,
    )
    return tsne.fit_transform(scaled)


def plot_tree_rep_tsne_for_folder(
    root_folder: Union[str, os.PathLike],
    tree_fields: Optional[Union[str, Sequence[str]]] = "tree_reps",
    source: str = "tree_reps",
    perplexity: int = 35,
    random_state: int = 0,
    stride: int = 1,
    max_files: Optional[int] = None,
) -> dict:
    """High-level helper that returns the embedding table and renders the chart."""
    dataset = build_tree_rep_dataset(
        root_folder=root_folder,
        tree_fields=tree_fields,
        source=source,
        max_files=max_files,
        stride=stride,
    )
    emb = fit_tsne(dataset["tree_matrix"], perplexity=perplexity, random_state=random_state)
    colors, base_color_map = _compute_point_colors(dataset["action_ids"], dataset["training_steps"])

    if tree_fields is None:
        fields_label = "all fields"
    elif isinstance(tree_fields, (list, tuple)):
        fields_label = " + ".join(tree_fields)
    else:
        fields_label = str(tree_fields)

    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(emb[:, 0], emb[:, 1], c=colors, s=18, linewidths=0, alpha=0.95)
    ax.set_title(f"t-SNE [{source}] ({fields_label})\nHue = action, brightness = training step")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")

    action_handles = [
        Line2D([0], [0], marker="o", linestyle="", color=base_color_map[action], label=f"{IDX_TO_ACTION.get(action, action)} ({action})")
        for action in sorted(base_color_map)
    ]
    legend_actions = ax.legend(
        handles=action_handles,
        title="Action (base hue)",
        loc="upper right",
        bbox_to_anchor=(1.32, 1.0),
    )
    ax.add_artist(legend_actions)

    unique_steps = np.unique(dataset["training_steps"])
    sample_steps = unique_steps
    if unique_steps.size > 5:
        quantiles = np.linspace(0.0, 1.0, num=5)
        sample_steps = np.unique(np.round(np.quantile(unique_steps, quantiles)).astype(int))

    def _strength_for_step(step_val: float) -> float:
        min_step = float(unique_steps.min())
        max_step = float(unique_steps.max())
        if math.isclose(min_step, max_step):
            return 0.35
        norm = (step_val - min_step) / (max_step - min_step)
        return 0.15 + 0.7 * norm

    step_handles = [
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="",
            color=_mix_with_white("black", _strength_for_step(step)),
            label=_format_training_step(float(step)),
        )
        for step in sample_steps
    ]
    ax.legend(
        handles=step_handles,
        title="Training step\n(brightness only)",
        loc="lower right",
        bbox_to_anchor=(1.32, 0.0),
    )

    plt.tight_layout()
    plt.show()

    tsne_table = {
        "tsne_1": emb[:, 0],
        "tsne_2": emb[:, 1],
        "action_id": dataset["action_ids"],
        "action_label": dataset["action_labels"],
        "training_step": dataset["training_steps"],
        "planning_depth": dataset["planning_depths"],
        "source_file": dataset["source_files"],
        "real_index": dataset["real_indices"],
    }

    return tsne_table


def plot_multiple_tsne_targets(
    root_folder: Union[str, os.PathLike],
    configs: Optional[Sequence[dict]] = None,
    shared_kwargs: Optional[dict] = None,
) -> dict:
    """Render several t-SNE plots (tree_reps / real_imgs / real_vectors) sequentially."""
    shared_kwargs = dict(shared_kwargs) if shared_kwargs else {}
    if configs is None:
        configs = [
            {"name": "tree_reps", "source": "tree_reps"},
            {"name": "real_imgs", "source": "real_imgs"},
            {"name": "real_vectors", "source": "real_vectors"},
        ]

    results = {}
    for cfg in configs:
        cfg = cfg.copy()
        name = cfg.pop("name", cfg.get("source", "tree_reps"))
        print("=" * 80)
        print(f"[t-SNE] {name}")
        local_kwargs = {**shared_kwargs, **cfg}
        results[name] = plot_tree_rep_tsne_for_folder(
            root_folder=root_folder,
            **local_kwargs,
        )

    return results


In [ ]:
tsne_tables = plot_multiple_tsne_targets(
    root_folder='test/spaceinvaderscheckpoint',
    shared_kwargs=dict(perplexity=35, stride=4, max_files=None, random_state=0),
    # optional: override defaults
    configs=[
        {"name": "tree_reps", "source": "tree_reps"},
        {"name": "real_imgs", "source": "real_imgs"},
        {"name": "real_vectors", "source": "real_vectors"},
    ],
)

[t-SNE] tree_reps
[loaded] spaceinvaders_10e6_0.npy            real_steps=  555  step=10.0M  source=tree_reps
[loaded] spaceinvaders_10e6_1.npy            real_steps=  892  step=10.0M  source=tree_reps


In [3]:
file_dir = 'spaceinvaders_1e6_0.npz'
data_dir = os.path.join('./spaceinvadersresult', file_dir)
data = (np.load(data_dir, allow_pickle=True))

In [6]:
data['action']

array([0, 0, 1, ..., 1, 1, 0], dtype=int32)